# Phase 6.5 — Controlled Tuning of Semantic + Reranker Retrieval

This notebook tunes the selected Phase 6 configuration:

**Semantic Retrieval + Cross-Encoder Reranker**

Only these parameters are changed:
- Semantic candidate Top-K
- Final reranked Top-K

An 80/20 deterministic development/holdout split is used. The holdout set is evaluated only after the best configuration is selected.

In [1]:
import json
import time
import random
import importlib.util
from pathlib import Path

import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer, CrossEncoder
from IPython.display import display

## Configuration

In [5]:
BASE_DIR = Path("content")

CHUNKS_FILE = BASE_DIR / "chunked_data" / "chunks.jsonl"
SHORT_QUERIES_FILE = BASE_DIR / "queries" / "short_queries_116.py"
LONG_QUERIES_FILE = BASE_DIR / "queries" / "long_queries_41.py"

OUTPUT_DIR = BASE_DIR / "phase6_5_tuning_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EMBEDDING_MODEL_NAME = "BAAI/bge-small-en-v1.5"
RERANKER_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"

RANDOM_SEED = 42
DEV_RATIO = 0.80

SEMANTIC_TOP_K_VALUES = [5, 10, 15, 20, 30, 40]
FINAL_TOP_K_VALUES = [3, 5]

# Phase 6 baseline C
BASELINE_SEMANTIC_TOP_K = 20
BASELINE_FINAL_TOP_K = 5

print("Configuration loaded.")
print("Configuration loaded.")

print()

print("Embedding model:", EMBEDDING_MODEL_NAME)

print("Reranker:", RERANKER_MODEL_NAME)

print("Semantic Top-K values:", SEMANTIC_TOP_K_VALUES)

print("Final Top-K values:", FINAL_TOP_K_VALUES)

print("Development ratio:", DEV_RATIO)

print("Random seed:", RANDOM_SEED)

Configuration loaded.
Configuration loaded.

Embedding model: BAAI/bge-small-en-v1.5
Reranker: cross-encoder/ms-marco-MiniLM-L-6-v2
Semantic Top-K values: [5, 10, 15, 20, 30, 40]
Final Top-K values: [3, 5]
Development ratio: 0.8
Random seed: 42


## Verify files

In [7]:
required_files = [

CHUNKS_FILE,

SHORT_QUERIES_FILE,

LONG_QUERIES_FILE,

]



for path in required_files:

  print(f"{path}: "f"{'FOUND' if path.exists() else 'NOT FOUND'}")



assert CHUNKS_FILE.exists(), f"Missing: {CHUNKS_FILE}"

assert SHORT_QUERIES_FILE.exists(), f"Missing: {SHORT_QUERIES_FILE}"

assert LONG_QUERIES_FILE.exists(), f"Missing: {LONG_QUERIES_FILE}"



print()

print("✓ All required files found.")

content/chunked_data/chunks.jsonl: FOUND
content/queries/short_queries_116.py: FOUND
content/queries/long_queries_41.py: FOUND

✓ All required files found.


## Load Python query files

In [8]:
def load_python_variable(file_path, variable_name):
    spec = importlib.util.spec_from_file_location("query_module", file_path)
    if spec is None or spec.loader is None:
        raise ImportError(f"Could not load Python file: {file_path}")

    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)

    if not hasattr(module, variable_name):
        raise AttributeError(
            f"Variable '{variable_name}' not found in {file_path}"
        )

    return getattr(module, variable_name)


short_queries = load_python_variable(
    SHORT_QUERIES_FILE,
    "evaluation_queries"
)

long_queries = load_python_variable(
    LONG_QUERIES_FILE,
    "evaluation_queries_long"
)

print("Short queries:", len(short_queries))
print("Long queries:", len(long_queries))

assert len(short_queries) == 116
assert len(long_queries) == 41

Short queries: 116
Long queries: 41


## Load the 96-chunk corpus

In [9]:
chunks = []

with open(CHUNKS_FILE, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            chunks.append(json.loads(line))

print("Total chunks:", len(chunks))
assert len(chunks) == 96

chunk_texts = [chunk["content"] for chunk in chunks]
chunk_ids = [chunk["chunk_id"] for chunk in chunks]

print("Unique chunk IDs:", len(set(chunk_ids)))
assert len(set(chunk_ids)) == 96

Total chunks: 96
Unique chunk IDs: 96


## Load embedding model and generate normalized chunk embeddings

In [10]:
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

chunk_embeddings = embedding_model.encode(
    chunk_texts,
    normalize_embeddings=True,
    show_progress_bar=True
)

chunk_embeddings = np.asarray(chunk_embeddings)

print("Embedding shape:", chunk_embeddings.shape)
assert chunk_embeddings.shape[0] == 96

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Embedding shape: (96, 384)


## Semantic retrieval

In [11]:
def semantic_retrieve(query, top_k=20):
    top_k = min(top_k, len(chunks))

    query_embedding = embedding_model.encode(
        query,
        normalize_embeddings=True
    )
    query_embedding = np.asarray(query_embedding)

    # Normalized embeddings + dot product = cosine similarity
    scores = np.dot(chunk_embeddings, query_embedding)

    top_indices = np.argsort(scores)[::-1][:top_k]

    results = []

    for rank, idx in enumerate(top_indices, start=1):
        chunk = chunks[idx]

        results.append({
            "chunk_id": chunk["chunk_id"],
            "document": chunk["document"],
            "file_path": chunk.get("file_path"),
            "category": chunk.get("category"),
            "section_path": chunk.get("section_path"),
            "section_title": chunk.get("section_title"),
            "content": chunk["content"],
            "semantic_score": float(scores[idx]),
            "candidate_rank": rank
        })

    return results

## Load Cross-Encoder reranker

In [12]:
reranker = CrossEncoder(RERANKER_MODEL_NAME)
print("Reranker loaded:", RERANKER_MODEL_NAME)

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Reranker loaded: cross-encoder/ms-marco-MiniLM-L-6-v2


## Semantic + reranker retrieval

In [13]:
def semantic_reranker_retrieve(
    query,
    semantic_top_k=20,
    final_top_k=5
):
    candidates = semantic_retrieve(
        query,
        top_k=semantic_top_k
    )

    if not candidates:
        return []

    final_top_k = min(final_top_k, len(candidates))

    pairs = [
        [query, item["content"]]
        for item in candidates
    ]

    scores = reranker.predict(pairs)

    for item, score in zip(candidates, scores):
        item["rerank_score"] = float(score)

    candidates.sort(
        key=lambda x: x["rerank_score"],
        reverse=True
    )

    final_results = []

    for rank, item in enumerate(
        candidates[:final_top_k],
        start=1
    ):
        item = dict(item)
        item["rank"] = rank
        final_results.append(item)

    return final_results

## Deterministic 80/20 development/holdout split

In [14]:
def split_queries(queries, dev_ratio=0.80, seed=42):
    queries = list(queries)

    rng = random.Random(seed)
    indices = list(range(len(queries)))
    rng.shuffle(indices)

    dev_size = int(len(queries) * dev_ratio)

    dev_indices = indices[:dev_size]
    holdout_indices = indices[dev_size:]

    dev_queries = [queries[i] for i in dev_indices]
    holdout_queries = [queries[i] for i in holdout_indices]

    return dev_queries, holdout_queries


short_dev, short_holdout = split_queries(
    short_queries, DEV_RATIO, RANDOM_SEED
)

long_dev, long_holdout = split_queries(
    long_queries, DEV_RATIO, RANDOM_SEED
)

print("Short:", len(short_dev), "dev /", len(short_holdout), "holdout")
print("Long :", len(long_dev), "dev /", len(long_holdout), "holdout")

Short: 92 dev / 24 holdout
Long : 32 dev / 9 holdout


## Verify split has no overlap

In [15]:
def query_keys(query_list):
    return {
        item.get("query_id", item["query"])
        for item in query_list
    }

assert query_keys(short_dev).isdisjoint(query_keys(short_holdout))
assert query_keys(long_dev).isdisjoint(query_keys(long_holdout))

print("✓ Development and holdout sets are disjoint.")

✓ Development and holdout sets are disjoint.


## Evaluation helpers

In [16]:
def get_expected_document(query_item):
    if "expected_document" in query_item:
        return query_item["expected_document"]

    if "expected" in query_item:
        return query_item["expected"]

    raise KeyError("Could not find expected document field.")


def evaluate_results(results):
    total = len(results)

    top1 = 0
    top3 = 0
    top5 = 0

    details = []

    for item in results:
        # IMPORTANT:
        # run_configuration stores this as expected_document.
        expected = item["expected_document"]

        retrieved_documents = [
            result["document"]
            for result in item["results"]
        ]

        rank = 999

        for i, document in enumerate(
            retrieved_documents,
            start=1
        ):
            if document == expected:
                rank = i
                break

        if rank <= 1:
            top1 += 1
        if rank <= 3:
            top3 += 1
        if rank <= 5:
            top5 += 1

        details.append({
            "query_id": item["query_id"],
            "query": item["query"],
            "expected_document": expected,
            "rank": rank,
            "top1": rank <= 1,
            "top3": rank <= 3,
            "top5": rank <= 5
        })

    metrics = {
        "total_queries": total,
        "top1_accuracy": top1 / total if total else 0,
        "top3_recall": top3 / total if total else 0,
        "top5_recall": top5 / total if total else 0,
        "top1_correct": top1,
        "top3_correct": top3,
        "top5_correct": top5
    }

    return metrics, details

## Run one tuning configuration

In [17]:
def run_configuration(
    queries,
    semantic_top_k,
    final_top_k,
    dataset_name="dataset"
):
    output = []
    start_time = time.time()

    for i, query_item in enumerate(queries, start=1):
        query = query_item["query"]

        expected = get_expected_document(
            query_item
        )

        results = semantic_reranker_retrieve(
            query,
            semantic_top_k=semantic_top_k,
            final_top_k=final_top_k
        )

        output.append({
            "query_id": query_item.get("query_id", i),
            "query": query,
            "query_type": query_item.get("query_type"),
            "expected_document": expected,
            "results": results
        })

        print(
            f"\r{dataset_name}: {i}/{len(queries)}",
            end=""
        )

    print()

    return output, time.time() - start_time

## Build valid 12-configuration grid

In [18]:
# Every requested combination is valid here because
# final_top_k <= semantic_top_k for all combinations.

tuning_configurations = [
    {
        "semantic_top_k": semantic_top_k,
        "final_top_k": final_top_k
    }
    for semantic_top_k in SEMANTIC_TOP_K_VALUES
    for final_top_k in FINAL_TOP_K_VALUES
    if final_top_k <= semantic_top_k
]

print("Configurations:", len(tuning_configurations))
display(pd.DataFrame(tuning_configurations))

Configurations: 12


,semantic_top_k,final_top_k
0,5,3
1,5,5
2,10,3
3,10,5
4,15,3
5,15,5
6,20,3
7,20,5
8,30,3
9,30,5


## Tune on short development queries

In [19]:
short_tuning_results = []
short_raw_results = {}

for config in tuning_configurations:
    k = config["semantic_top_k"]
    n = config["final_top_k"]

    label = f"semantic_{k}_final_{n}"

    print("\n" + "=" * 70)
    print("SHORT DEV:", label)
    print("=" * 70)

    results, elapsed = run_configuration(
        short_dev,
        k,
        n,
        dataset_name=label
    )

    metrics, details = evaluate_results(results)
    short_raw_results[label] = results

    short_tuning_results.append({
        "semantic_top_k": k,
        "final_top_k": n,
        "top1_accuracy": metrics["top1_accuracy"],
        "top3_recall": metrics["top3_recall"],
        "top5_recall": metrics["top5_recall"],
        "top1_correct": metrics["top1_correct"],
        "top3_correct": metrics["top3_correct"],
        "top5_correct": metrics["top5_correct"],
        "runtime_seconds": elapsed
    })

short_tuning_df = pd.DataFrame(short_tuning_results)

print("\nSHORT DEVELOPMENT RESULTS")
display(short_tuning_df.round(4))


SHORT DEV: semantic_5_final_3
semantic_5_final_3: 92/92

SHORT DEV: semantic_5_final_5
semantic_5_final_5: 92/92

SHORT DEV: semantic_10_final_3
semantic_10_final_3: 92/92

SHORT DEV: semantic_10_final_5
semantic_10_final_5: 92/92

SHORT DEV: semantic_15_final_3
semantic_15_final_3: 92/92

SHORT DEV: semantic_15_final_5
semantic_15_final_5: 92/92

SHORT DEV: semantic_20_final_3
semantic_20_final_3: 92/92

SHORT DEV: semantic_20_final_5
semantic_20_final_5: 92/92

SHORT DEV: semantic_30_final_3
semantic_30_final_3: 92/92

SHORT DEV: semantic_30_final_5
semantic_30_final_5: 92/92

SHORT DEV: semantic_40_final_3
semantic_40_final_3: 92/92

SHORT DEV: semantic_40_final_5
semantic_40_final_5: 92/92

SHORT DEVELOPMENT RESULTS


,semantic_top_k,final_top_k,top1_accuracy,top3_recall,top5_recall,top1_correct,top3_correct,top5_correct,runtime_seconds
0,5,3,0.8478,0.9130,0.9130,78,84,84,2.8325
1,5,5,0.8478,0.9130,0.9130,78,84,84,2.5676
2,10,3,0.8587,0.9022,0.9022,79,83,83,4.6686
3,10,5,0.8587,0.9022,0.9348,79,83,86,4.5537
4,15,3,0.8587,0.9022,0.9022,79,83,83,6.3153
5,15,5,0.8587,0.9022,0.9239,79,83,85,6.8099
6,20,3,0.8587,0.9022,0.9022,79,83,83,8.8782
7,20,5,0.8587,0.9022,0.9130,79,83,84,8.5062
8,30,3,0.8587,0.8913,0.8913,79,82,82,13.5903
9,30,5,0.8587,0.8913,0.9130,79,82,84,13.8645


## Tune on long development queries

In [20]:
long_tuning_results = []
long_raw_results = {}

for config in tuning_configurations:
    k = config["semantic_top_k"]
    n = config["final_top_k"]

    label = f"semantic_{k}_final_{n}"

    print("\n" + "=" * 70)
    print("LONG DEV:", label)
    print("=" * 70)

    results, elapsed = run_configuration(
        long_dev,
        k,
        n,
        dataset_name=label
    )

    metrics, details = evaluate_results(results)
    long_raw_results[label] = results

    long_tuning_results.append({
        "semantic_top_k": k,
        "final_top_k": n,
        "top1_accuracy": metrics["top1_accuracy"],
        "top3_recall": metrics["top3_recall"],
        "top5_recall": metrics["top5_recall"],
        "top1_correct": metrics["top1_correct"],
        "top3_correct": metrics["top3_correct"],
        "top5_correct": metrics["top5_correct"],
        "runtime_seconds": elapsed
    })

long_tuning_df = pd.DataFrame(long_tuning_results)

print("\nLONG DEVELOPMENT RESULTS")
display(long_tuning_df.round(4))


LONG DEV: semantic_5_final_3
semantic_5_final_3: 32/32

LONG DEV: semantic_5_final_5
semantic_5_final_5: 32/32

LONG DEV: semantic_10_final_3
semantic_10_final_3: 32/32

LONG DEV: semantic_10_final_5
semantic_10_final_5: 32/32

LONG DEV: semantic_15_final_3
semantic_15_final_3: 32/32

LONG DEV: semantic_15_final_5
semantic_15_final_5: 32/32

LONG DEV: semantic_20_final_3
semantic_20_final_3: 32/32

LONG DEV: semantic_20_final_5
semantic_20_final_5: 32/32

LONG DEV: semantic_30_final_3
semantic_30_final_3: 32/32

LONG DEV: semantic_30_final_5
semantic_30_final_5: 32/32

LONG DEV: semantic_40_final_3
semantic_40_final_3: 32/32

LONG DEV: semantic_40_final_5
semantic_40_final_5: 32/32

LONG DEVELOPMENT RESULTS


,semantic_top_k,final_top_k,top1_accuracy,top3_recall,top5_recall,top1_correct,top3_correct,top5_correct,runtime_seconds
0,5,3,0.9062,0.9062,0.9062,29,29,29,1.1163
1,5,5,0.9062,0.9062,0.9375,29,29,30,1.1789
2,10,3,0.9062,0.9688,0.9688,29,31,31,2.0470
3,10,5,0.9062,0.9688,0.9688,29,31,31,1.6496
4,15,3,0.9062,0.9688,0.9688,29,31,31,2.4169
5,15,5,0.9062,0.9688,0.9688,29,31,31,2.4114
6,20,3,0.9062,0.9688,0.9688,29,31,31,3.2756
7,20,5,0.9062,0.9688,0.9688,29,31,31,3.7201
8,30,3,0.9062,0.9688,0.9688,29,31,31,5.0508
9,30,5,0.9062,0.9688,0.9688,29,31,31,5.3552


## Select the best configuration

In [21]:
short_lookup = short_tuning_df.set_index(
    ["semantic_top_k", "final_top_k"]
)

long_lookup = long_tuning_df.set_index(
    ["semantic_top_k", "final_top_k"]
)

selection_rows = []

for config in tuning_configurations:
    k = config["semantic_top_k"]
    n = config["final_top_k"]

    s = short_lookup.loc[(k, n)]
    l = long_lookup.loc[(k, n)]

    mean_top1 = np.mean([
        s["top1_accuracy"],
        l["top1_accuracy"]
    ])

    mean_top3 = np.mean([
        s["top3_recall"],
        l["top3_recall"]
    ])

    mean_top5 = np.mean([
        s["top5_recall"],
        l["top5_recall"]
    ])

    # Primary: Top-1
    # Secondary: Top-3
    # Tertiary: Top-5
    selection_score = (
        0.60 * mean_top1
        + 0.25 * mean_top3
        + 0.15 * mean_top5
    )

    selection_rows.append({
        "semantic_top_k": k,
        "final_top_k": n,
        "short_top1": s["top1_accuracy"],
        "long_top1": l["top1_accuracy"],
        "mean_top1": mean_top1,
        "mean_top3": mean_top3,
        "mean_top5": mean_top5,
        "selection_score": selection_score
    })

selection_df = pd.DataFrame(selection_rows).sort_values(
    ["selection_score", "mean_top1", "mean_top3", "mean_top5"],
    ascending=False
).reset_index(drop=True)

display(selection_df.round(4))

best_config = selection_df.iloc[0]

BEST_SEMANTIC_TOP_K = int(best_config["semantic_top_k"])
BEST_FINAL_TOP_K = int(best_config["final_top_k"])

print("Selected configuration:")
print("Semantic Top-K:", BEST_SEMANTIC_TOP_K)
print("Final Top-K:", BEST_FINAL_TOP_K)
print("Mean Top-1:", round(best_config["mean_top1"] * 100, 2), "%")
print("Mean Top-3:", round(best_config["mean_top3"] * 100, 2), "%")
print("Mean Top-5:", round(best_config["mean_top5"] * 100, 2), "%")
print("Selection score:", round(best_config["selection_score"], 4))

,semantic_top_k,final_top_k,short_top1,long_top1,mean_top1,mean_top3,mean_top5,selection_score
0,10,5,0.8587,0.9062,0.8825,0.9355,0.9518,0.9061
1,15,5,0.8587,0.9062,0.8825,0.9355,0.9463,0.9053
2,20,5,0.8587,0.9062,0.8825,0.9355,0.9409,0.9045
3,10,3,0.8587,0.9062,0.8825,0.9355,0.9355,0.9037
4,15,3,0.8587,0.9062,0.8825,0.9355,0.9355,0.9037
5,20,3,0.8587,0.9062,0.8825,0.9355,0.9355,0.9037
6,30,5,0.8587,0.9062,0.8825,0.9300,0.9409,0.9031
7,40,5,0.8587,0.9062,0.8825,0.9300,0.9409,0.9031
8,30,3,0.8587,0.9062,0.8825,0.9300,0.9300,0.9015
9,40,3,0.8587,0.9062,0.8825,0.9300,0.9300,0.9015


Selected configuration:
Semantic Top-K: 10
Final Top-K: 5
Mean Top-1: 88.25 %
Mean Top-3: 93.55 %
Mean Top-5: 95.18 %
Selection score: 0.9061


## Compare tuned configuration with Phase 6 baseline C

In [22]:
baseline_rows = selection_df[
    (selection_df["semantic_top_k"] == BASELINE_SEMANTIC_TOP_K)
    & (selection_df["final_top_k"] == BASELINE_FINAL_TOP_K)
]

if len(baseline_rows) == 1:
    baseline = baseline_rows.iloc[0]

    comparison = pd.DataFrame([
        {
            "Configuration": "Phase 6 Baseline C",
            "Semantic Top-K": BASELINE_SEMANTIC_TOP_K,
            "Final Top-K": BASELINE_FINAL_TOP_K,
            "Mean Top-1": baseline["mean_top1"],
            "Mean Top-3": baseline["mean_top3"],
            "Mean Top-5": baseline["mean_top5"],
            "Selection Score": baseline["selection_score"]
        },
        {
            "Configuration": "Phase 6.5 Tuned C",
            "Semantic Top-K": BEST_SEMANTIC_TOP_K,
            "Final Top-K": BEST_FINAL_TOP_K,
            "Mean Top-1": best_config["mean_top1"],
            "Mean Top-3": best_config["mean_top3"],
            "Mean Top-5": best_config["mean_top5"],
            "Selection Score": best_config["selection_score"]
        }
    ])

    display(comparison.round(4))

    print("Top-1 Δ:",
          round((best_config["mean_top1"] - baseline["mean_top1"]) * 100, 2),
          "percentage points")
    print("Top-3 Δ:",
          round((best_config["mean_top3"] - baseline["mean_top3"]) * 100, 2),
          "percentage points")
    print("Top-5 Δ:",
          round((best_config["mean_top5"] - baseline["mean_top5"]) * 100, 2),
          "percentage points")

,Configuration,Semantic Top-K,Final Top-K,Mean Top-1,Mean Top-3,Mean Top-5,Selection Score
0,Phase 6 Baseline C,20,5,0.8825,0.9355,0.9409,0.9045
1,Phase 6.5 Tuned C,10,5,0.8825,0.9355,0.9518,0.9061


Top-1 Δ: 0.0 percentage points
Top-3 Δ: 0.0 percentage points
Top-5 Δ: 1.09 percentage points


## Final evaluation on untouched holdout

In [23]:
print("=" * 70)
print("FINAL HOLDOUT — SHORT")
print("=" * 70)

short_holdout_results, short_holdout_time = run_configuration(
    short_holdout,
    BEST_SEMANTIC_TOP_K,
    BEST_FINAL_TOP_K,
    dataset_name="SHORT HOLDOUT"
)

short_holdout_metrics, short_holdout_details = evaluate_results(
    short_holdout_results
)

print(short_holdout_metrics)

print()
print("=" * 70)
print("FINAL HOLDOUT — LONG")
print("=" * 70)

long_holdout_results, long_holdout_time = run_configuration(
    long_holdout,
    BEST_SEMANTIC_TOP_K,
    BEST_FINAL_TOP_K,
    dataset_name="LONG HOLDOUT"
)

long_holdout_metrics, long_holdout_details = evaluate_results(
    long_holdout_results
)

print(long_holdout_metrics)

FINAL HOLDOUT — SHORT
SHORT HOLDOUT: 24/24
{'total_queries': 24, 'top1_accuracy': 0.75, 'top3_recall': 0.8333333333333334, 'top5_recall': 0.8333333333333334, 'top1_correct': 18, 'top3_correct': 20, 'top5_correct': 20}

FINAL HOLDOUT — LONG
LONG HOLDOUT: 9/9
{'total_queries': 9, 'top1_accuracy': 0.8888888888888888, 'top3_recall': 1.0, 'top5_recall': 1.0, 'top1_correct': 8, 'top3_correct': 9, 'top5_correct': 9}


## Final holdout summary

In [24]:
final_holdout_df = pd.DataFrame([
    {
        "Dataset": "Short Holdout",
        "Queries": len(short_holdout),
        "Semantic Top-K": BEST_SEMANTIC_TOP_K,
        "Final Top-K": BEST_FINAL_TOP_K,
        "Top-1 (%)": short_holdout_metrics["top1_accuracy"] * 100,
        "Top-3 (%)": short_holdout_metrics["top3_recall"] * 100,
        "Top-5 (%)": short_holdout_metrics["top5_recall"] * 100
    },
    {
        "Dataset": "Long Holdout",
        "Queries": len(long_holdout),
        "Semantic Top-K": BEST_SEMANTIC_TOP_K,
        "Final Top-K": BEST_FINAL_TOP_K,
        "Top-1 (%)": long_holdout_metrics["top1_accuracy"] * 100,
        "Top-3 (%)": long_holdout_metrics["top3_recall"] * 100,
        "Top-5 (%)": long_holdout_metrics["top5_recall"] * 100
    }
])

display(final_holdout_df.round(2))

final_mean_top1 = np.mean([
    short_holdout_metrics["top1_accuracy"],
    long_holdout_metrics["top1_accuracy"]
])

final_mean_top3 = np.mean([
    short_holdout_metrics["top3_recall"],
    long_holdout_metrics["top3_recall"]
])

final_mean_top5 = np.mean([
    short_holdout_metrics["top5_recall"],
    long_holdout_metrics["top5_recall"]
])

print("Mean Top-1:", round(final_mean_top1 * 100, 2), "%")
print("Mean Top-3:", round(final_mean_top3 * 100, 2), "%")
print("Mean Top-5:", round(final_mean_top5 * 100, 2), "%")

,Dataset,Queries,Semantic Top-K,Final Top-K,Top-1 (%),Top-3 (%),Top-5 (%)
0,Short Holdout,24,10,5,75.00,83.33,83.33
1,Long Holdout,9,10,5,88.89,100.00,100.00


Mean Top-1: 81.94 %
Mean Top-3: 91.67 %
Mean Top-5: 91.67 %


## Holdout error analysis

In [25]:
short_holdout_errors = pd.DataFrame(short_holdout_details)
short_holdout_errors = short_holdout_errors[
    short_holdout_errors["rank"] > 1
].sort_values("rank")

long_holdout_errors = pd.DataFrame(long_holdout_details)
long_holdout_errors = long_holdout_errors[
    long_holdout_errors["rank"] > 1
].sort_values("rank")

print("Short holdout Top-1 failures:", len(short_holdout_errors))
display(
    short_holdout_errors[
        ["query_id", "query", "expected_document", "rank"]
    ]
)

print("Long holdout Top-1 failures:", len(long_holdout_errors))
display(
    long_holdout_errors[
        ["query_id", "query", "expected_document", "rank"]
    ]
)

Short holdout Top-1 failures: 6


,query_id,query,expected_document,rank
23,24,What should I expect on my first day at Clef?,Welcome to Clef.md,2
19,20,How much do the founders make?,Salary and Equity Compensation.md,3
1,2,How does Clef think about inclusion?,Clef Values.md,999
5,6,What are the budget categories at Clef?,Budgeting.md,999
12,13,What are the meeting time requirements at Clef?,Effective Meetings.md,999
9,10,What are Clef's core values?,Clef Values.md,999


Long holdout Top-1 failures: 1


,query_id,query,expected_document,rank
6,7,My spouse and I are expecting a baby in a few ...,New Parent Leave.md,2


## Save all important outputs

In [26]:
def save_json(data, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(
            data,
            f,
            indent=2,
            ensure_ascii=False
        )


short_tuning_df.to_csv(
    OUTPUT_DIR / "short_development_tuning_results.csv",
    index=False
)

long_tuning_df.to_csv(
    OUTPUT_DIR / "long_development_tuning_results.csv",
    index=False
)

selection_df.to_csv(
    OUTPUT_DIR / "configuration_ranking.csv",
    index=False
)

final_holdout_df.to_csv(
    OUTPUT_DIR / "final_holdout_results.csv",
    index=False
)

short_holdout_errors.to_csv(
    OUTPUT_DIR / "short_holdout_errors.csv",
    index=False
)

long_holdout_errors.to_csv(
    OUTPUT_DIR / "long_holdout_errors.csv",
    index=False
)

save_json(
    short_holdout_results,
    OUTPUT_DIR / "short_holdout_results.json"
)

save_json(
    long_holdout_results,
    OUTPUT_DIR / "long_holdout_results.json"
)

experiment_summary = {
    "embedding_model": EMBEDDING_MODEL_NAME,
    "reranker_model": RERANKER_MODEL_NAME,
    "corpus_chunks": len(chunks),
    "total_short_queries": len(short_queries),
    "total_long_queries": len(long_queries),
    "development_ratio": DEV_RATIO,
    "random_seed": RANDOM_SEED,
    "semantic_top_k_search_space": SEMANTIC_TOP_K_VALUES,
    "final_top_k_search_space": FINAL_TOP_K_VALUES,
    "baseline_configuration": {
        "semantic_top_k": BASELINE_SEMANTIC_TOP_K,
        "final_top_k": BASELINE_FINAL_TOP_K
    },
    "selected_configuration": {
        "semantic_top_k": BEST_SEMANTIC_TOP_K,
        "final_top_k": BEST_FINAL_TOP_K
    },
    "final_holdout_performance": {
        "mean_top1": float(final_mean_top1),
        "mean_top3": float(final_mean_top3),
        "mean_top5": float(final_mean_top5)
    }
}

save_json(
    experiment_summary,
    OUTPUT_DIR / "phase6_5_experiment_summary.json"
)

print("All Phase 6.5 outputs saved to:")
print(OUTPUT_DIR.resolve())

All Phase 6.5 outputs saved to:
/content/content/phase6_5_tuning_outputs


## Final validation

In [27]:
assert len(chunks) == 96
assert len(short_queries) == 116
assert len(long_queries) == 41

assert len(short_dev) + len(short_holdout) == 116
assert len(long_dev) + len(long_holdout) == 41

assert len(short_holdout_results) == len(short_holdout)
assert len(long_holdout_results) == len(long_holdout)

assert BEST_SEMANTIC_TOP_K in SEMANTIC_TOP_K_VALUES
assert BEST_FINAL_TOP_K in FINAL_TOP_K_VALUES

print("=" * 80)
print("FINAL PHASE 6.5 RESULT")
print("=" * 80)

print("Architecture: Semantic Retrieval + Cross-Encoder Reranker")
print("Semantic candidate Top-K:", BEST_SEMANTIC_TOP_K)
print("Final reranked Top-K:", BEST_FINAL_TOP_K)

print()
print(f"Mean Holdout Top-1: {final_mean_top1 * 100:.2f}%")
print(f"Mean Holdout Top-3: {final_mean_top3 * 100:.2f}%")
print(f"Mean Holdout Top-5: {final_mean_top5 * 100:.2f}%")

print()
print("✓ 96 clean corpus chunks")
print("✓ 116 short queries")
print("✓ 41 long queries")
print("✓ Deterministic 80/20 development/holdout split")
print("✓ Tuning performed only on development queries")
print("✓ Final configuration evaluated on untouched holdout")
print("✓ Phase 6.5 completed successfully")

FINAL PHASE 6.5 RESULT
Architecture: Semantic Retrieval + Cross-Encoder Reranker
Semantic candidate Top-K: 10
Final reranked Top-K: 5

Mean Holdout Top-1: 81.94%
Mean Holdout Top-3: 91.67%
Mean Holdout Top-5: 91.67%

✓ 96 clean corpus chunks
✓ 116 short queries
✓ 41 long queries
✓ Deterministic 80/20 development/holdout split
✓ Tuning performed only on development queries
✓ Final configuration evaluated on untouched holdout
✓ Phase 6.5 completed successfully


## Conclusion

Phase 6.5 evaluated controlled variations of the semantic retrieval and cross-encoder reranking pipeline. The configuration using 10 semantic candidates followed by 5 reranked passages achieved the highest development selection score (0.9061). Compared with the Phase 6 baseline, it maintained Top-1 and Top-3 performance while improving Top-5 recall from 94.09% to 95.18%. On the untouched holdout set, the selected configuration achieved 81.94% Top-1 accuracy, 91.67% Top-3 recall, and 91.67% Top-5 recall. This configuration was therefore selected as the final retrieval configuration for subsequent RAG generation experiments.